# Construction du dataset capybara

Ce notebook assemble le dataset du projet à partir de trois sources :

- les observations de six mammifères sud-américains exportées depuis GBIF (la table de base, une ligne par observation),
- le climat et l'altitude de WorldClim 2.1 (rasters),
- le réseau de rivières HydroRIVERS (lignes), pour calculer la distance à l'eau.

Il produit un seul fichier, `capybara_dataset.csv.gz`, à pousser sur GitHub. Le notebook principal du projet ne charge que ce fichier.

Les fichiers bruts sont lourds, ils restent sur Google Drive et ne vont pas sur GitHub. Lancez les cellules dans l'ordre. Les téléchargements ne se font qu'une fois, les fois suivantes les fichiers sont relus depuis Drive.

**À vérifier au premier lancement** : les URL de WorldClim et HydroRIVERS suivent les conventions des deux sites mais n'ont pas pu être testées. Si un téléchargement échoue, téléchargez le fichier à la main depuis le site et déposez-le dans le dossier indiqué.

## 1. Installation et imports

In [ ]:
!pip install -q rasterio pygbif

In [ ]:
import os, json, zipfile, glob, urllib.request
from datetime import date

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

print("pandas", pd.__version__, "| geopandas", gpd.__version__, "| rasterio", rasterio.__version__)

## 2. Configuration

Tout ce qui peut changer se règle ici : dossier de travail, espèces, seuils de nettoyage.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
BASE = "/content/drive/MyDrive/projet_capybara"   # dossier de travail sur Drive
RAW = f"{BASE}/raw"                               # fichiers bruts telecharges
OUT = f"{BASE}/output"                            # dataset final
os.makedirs(RAW, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

SPECIES = [
    "Hydrochoerus hydrochaeris",   # capybara
    "Tapirus terrestris",          # tapir terrestre
    "Nasua nasua",                 # coati roux
    "Myrmecophaga tridactyla",     # grand fourmilier
    "Blastocerus dichotomus",      # cerf des marais
    "Lontra longicaudis",          # loutre neotropicale
]

MAX_UNCERTAINTY_M = 10_000     # on ecarte les points plus imprecis que 10 km (seuil a justifier dans l'EDA)
MIN_YEAR = 1970                # coherent avec la periode climatique WorldClim (1970-2000), a discuter
WC_RES = "2.5m"                # resolution WorldClim (environ 5 km)
BIO_VARS = [1, 4, 12, 14]      # temperature moyenne, saisonnalite temperature, pluie annuelle, pluie mois le plus sec
METRIC_CRS = "ESRI:102033"     # Albers equivalente Amerique du Sud, pour des distances en metres
CELL_DEG = 1.0                 # taille des cellules pour la validation croisee spatiale

GITHUB_USER = "anthony-eluecque"
GITHUB_REPO = "imt-aaa-capybara-project"
GBIF_ZIP_NAME = "0005319-261110092616502.zip"
GBIF_DOI = "https://doi.org/10.15468/dl.ut27sj"

## 3. Observations GBIF

L'export a été fait à la main sur gbif.org, avec les filtres suivants :

- espèces : Hydrochoerus hydrochaeris, Tapirus terrestris, Nasua nasua, Myrmecophaga tridactyla, Blastocerus dichotomus, Lontra longicaudis,
- continent : Amérique du Sud,
- avec coordonnées, sans problème géospatial,
- statut d'occurrence : présent,
- format : Simple (fichier séparé par des tabulations).

Le zip reçu est stocké tel quel, sans modification, dans le dossier `raw/` du dépôt GitHub. Son DOI est renseigné dans `GBIF_DOI` (configuration) et doit être cité dans le notebook principal.

### Récupération depuis GitHub

Copie le zip GBIF du dépôt vers le dossier de travail. Le dépôt doit être public. La cellule ne fait rien si le zip est déjà là ou si la configuration n'est pas renseignée (cas de l'option B).

In [ ]:
url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/main/raw/{GBIF_ZIP_NAME}"
dest = f"{RAW}/{GBIF_ZIP_NAME}"
if not os.path.exists(dest):
    print("Telechargement depuis GitHub :", url)
    urllib.request.urlretrieve(url, dest)
print("Zip GBIF disponible :", dest)

### Lecture de l'export

In [ ]:
GBIF_ZIP = f"{RAW}/{GBIF_ZIP_NAME}"
print("Export utilise :", GBIF_ZIP)

COLS = ["gbifID", "species", "decimalLatitude", "decimalLongitude", "countryCode",
        "basisOfRecord", "individualCount", "coordinateUncertaintyInMeters",
        "month", "year", "eventDate"]

with zipfile.ZipFile(GBIF_ZIP) as z:
    csv_name = [n for n in z.namelist() if n.endswith((".csv", ".txt"))][0]
    with z.open(csv_name) as f:
        # le format Simple de GBIF est separe par des tabulations
        raw = pd.read_csv(f, sep="\t", usecols=lambda c: c in COLS,
                          low_memory=False, on_bad_lines="warn")

print(raw.shape)
raw["species"].value_counts()

## 4. Nettoyage

Chaque filtre est journalisé. Le tableau final sert directement dans l'EDA pour justifier les choix.

In [ ]:
log = []
def step(df, label):
    log.append({"etape": label, "lignes": len(df)})
    print(f"{label:55s} {len(df):>8,}")
    return df

df = step(raw.copy(), "Export brut")
df = step(df[df["species"].isin(SPECIES)], "Especes retenues uniquement")
df = step(df.dropna(subset=["decimalLatitude", "decimalLongitude"]), "Coordonnees presentes")
df = step(df[~((df.decimalLatitude == 0) & (df.decimalLongitude == 0))], "Sans le point 0,0")
df = step(df[df["coordinateUncertaintyInMeters"].isna() |
             (df["coordinateUncertaintyInMeters"] <= MAX_UNCERTAINTY_M)],
          f"Incertitude <= {MAX_UNCERTAINTY_M} m (ou non renseignee)")
df = step(df[df["year"].isna() | (df["year"] >= MIN_YEAR)], f"Annee >= {MIN_YEAR} (ou non renseignee)")
df = step(df.drop_duplicates(["species", "decimalLatitude", "decimalLongitude", "eventDate"]),
          "Sans doublons (espece, lieu, date)")
df = df.reset_index(drop=True)

pd.DataFrame(log)

## 5. Climat et altitude (WorldClim 2.1)

Chaque fichier WorldClim est une grille de pixels. On lit la valeur du pixel sous chaque observation, ce qui ajoute une colonne par variable.

In [ ]:
WC_BASE = "https://geodata.ucdavis.edu/climate/worldclim/2_1/base"   # URL a verifier
WC_DIR = f"{RAW}/worldclim"
os.makedirs(WC_DIR, exist_ok=True)

for name in [f"wc2.1_{WC_RES}_bio.zip", f"wc2.1_{WC_RES}_elev.zip"]:
    path = f"{RAW}/{name}"
    if not os.path.exists(path):
        print("Telechargement de", name, "...")
        urllib.request.urlretrieve(f"{WC_BASE}/{name}", path)
    with zipfile.ZipFile(path) as z:
        z.extractall(WC_DIR)

sorted(os.listdir(WC_DIR))[:5]

In [ ]:
def sample_raster(tif_path, lon, lat):
    """Valeur du pixel sous chaque point. Les pixels vides (mer, hors grille) deviennent NaN."""
    with rasterio.open(tif_path) as src:
        vals = np.array([v[0] for v in src.sample(zip(lon, lat))], dtype="float64")
        if src.nodata is not None:
            vals[vals == src.nodata] = np.nan
    vals[np.abs(vals) > 1e30] = np.nan   # certaines valeurs nodata ne sont pas declarees
    return vals

lon, lat = df["decimalLongitude"].values, df["decimalLatitude"].values
WC_NAMES = {1: "bio1_temp_mean_c", 4: "bio4_temp_seasonality",
            12: "bio12_precip_annual_mm", 14: "bio14_precip_driest_month_mm"}

for b in BIO_VARS:
    tif = glob.glob(f"{WC_DIR}/*_bio_{b}.tif")[0]
    df[WC_NAMES.get(b, f"bio{b}")] = sample_raster(tif, lon, lat)

df["elevation_m"] = sample_raster(glob.glob(f"{WC_DIR}/*_elev.tif")[0], lon, lat)

n_before = len(df)
print("Points sans valeur climatique (mer, cote) :", df["bio1_temp_mean_c"].isna().sum())
df[list(WC_NAMES.values()) + ["elevation_m"]].describe().T

## 6. Distance à la rivière la plus proche (HydroRIVERS)

On transforme les observations en points géographiques, puis on cherche la rivière la plus proche de chacune. Le calcul se fait dans une projection métrique pour obtenir des mètres et non des degrés.

Le fichier Amérique du Sud est volumineux. On ne charge que la zone couverte par les observations. Si c'est encore trop lent, réglez `MAX_ORD_FLOW` pour ne garder que les plus grandes rivières (dans HydroRIVERS, plus l'ordre de débit est petit, plus la rivière est grande). Ce choix change le sens de la variable, à documenter.

In [ ]:
HR_NAME = "HydroRIVERS_v10_sa_shp.zip"
HR_URL = f"https://data.hydrosheds.org/file/HydroRIVERS/{HR_NAME}"   # URL a verifier
HR_DIR = f"{RAW}/hydrorivers"
if not os.path.exists(f"{RAW}/{HR_NAME}"):
    print("Telechargement de HydroRIVERS ...")
    urllib.request.urlretrieve(HR_URL, f"{RAW}/{HR_NAME}")
with zipfile.ZipFile(f"{RAW}/{HR_NAME}") as z:
    z.extractall(HR_DIR)
HR_SHP = glob.glob(f"{HR_DIR}/**/*.shp", recursive=True)[0]
HR_SHP

In [ ]:
MAX_ORD_FLOW = None   # par exemple 6 pour ne garder que les grandes rivieres

pts = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.decimalLongitude, df.decimalLatitude),
                       crs="EPSG:4326")

bbox = tuple(pts.total_bounds + np.array([-1, -1, 1, 1]))   # zone des points + 1 degre de marge
rivers = gpd.read_file(HR_SHP, bbox=bbox)
if MAX_ORD_FLOW is not None and "ORD_FLOW" in rivers.columns:
    rivers = rivers[rivers["ORD_FLOW"] <= MAX_ORD_FLOW]
print(f"{len(rivers):,} troncons de riviere charges")

pts_m = pts.to_crs(METRIC_CRS)
rivers_m = rivers[["geometry"]].to_crs(METRIC_CRS)

joined = gpd.sjoin_nearest(pts_m, rivers_m, how="left", distance_col="dist_river_m")
joined = joined[~joined.index.duplicated(keep="first")]   # un point a egale distance de deux rivieres
df["dist_river_m"] = joined["dist_river_m"].reindex(df.index).values

assert len(df) == n_before, "La jointure a change le nombre de lignes"
df["dist_river_m"].describe()

## 7. Feature engineering

- **Saison** : dérivée du mois en tenant compte de l'hémisphère (l'été austral va de décembre à février).
- **Log de la distance à la rivière** : l'effet de l'eau compte surtout à courte distance.
- **Indice d'aridité de De Martonne** : précipitations annuelles / (température moyenne + 10). Plus il est bas, plus le milieu est sec.
- **Cellule spatiale** : pas une feature, mais le groupe utilisé pour la validation croisée spatiale.

In [ ]:
NORTH = {12: "hiver", 1: "hiver", 2: "hiver", 3: "printemps", 4: "printemps", 5: "printemps",
         6: "ete", 7: "ete", 8: "ete", 9: "automne", 10: "automne", 11: "automne"}
FLIP = {"hiver": "ete", "ete": "hiver", "printemps": "automne", "automne": "printemps"}

def season(month, lat):
    if pd.isna(month):
        return np.nan
    s = NORTH[int(month)]
    return FLIP[s] if lat < 0 else s

df["season"] = [season(m, la) for m, la in zip(df["month"], df["decimalLatitude"])]
df["log_dist_river"] = np.log1p(df["dist_river_m"])
df["aridity_de_martonne"] = df["bio12_precip_annual_mm"] / (df["bio1_temp_mean_c"] + 10)

df["spatial_cell"] = (np.floor(df.decimalLatitude / CELL_DEG).astype(int).astype(str) + "_" +
                      np.floor(df.decimalLongitude / CELL_DEG).astype(int).astype(str))

print("Cellules spatiales :", df["spatial_cell"].nunique())
df[["season", "log_dist_river", "aridity_de_martonne", "spatial_cell"]].head()

## 8. Contrôles avant export

In [ ]:
final = df.rename(columns={
    "decimalLatitude": "latitude", "decimalLongitude": "longitude",
    "countryCode": "country", "basisOfRecord": "basis_of_record",
    "individualCount": "individual_count",
    "coordinateUncertaintyInMeters": "coordinate_uncertainty_m",
})
final = final[["gbifID", "species", "latitude", "longitude", "country", "basis_of_record",
               "individual_count", "coordinate_uncertainty_m", "month", "year", "season",
               *WC_NAMES.values(), "elevation_m", "dist_river_m", "log_dist_river",
               "aridity_de_martonne", "spatial_cell"]]

print("Dimensions :", final.shape)
print("\nRepartition des classes :")
print(final["species"].value_counts())
print("\nValeurs manquantes (%) :")
print((final.isna().mean() * 100).round(1).sort_values(ascending=False).head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
for sp, g in final.groupby("species"):
    ax.scatter(g.longitude, g.latitude, s=2, alpha=0.5, label=sp)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Observations par espece")
ax.legend(markerscale=5, fontsize=8)
plt.show()

## 9. Export

On écrit le CSV compressé et un petit fichier de métadonnées (DOI GBIF, date, journal du nettoyage). Ensuite, copiez les deux fichiers dans le dossier `data/` du dépôt GitHub. Dans le notebook principal, le chargement tient en une ligne :

```python
df = pd.read_csv("https://raw.githubusercontent.com/<compte>/<depot>/main/data/capybara_dataset.csv.gz")
```

Les observations avec des valeurs manquantes sont gardées volontairement. Leur traitement fait partie de l'EDA.

In [ ]:
final.to_csv(f"{OUT}/capybara_dataset.csv.gz", index=False, compression="gzip")

gbif_meta = {"zip": GBIF_ZIP_NAME, "doi": GBIF_DOI}
with open(f"{OUT}/dataset_metadata.json", "w") as f:
    json.dump({"built_on": str(date.today()), "gbif": gbif_meta,
               "worldclim": f"WorldClim 2.1, {WC_RES}, bio {BIO_VARS} + elevation",
               "hydrorivers": "HydroRIVERS v1.0, South America",
               "max_ord_flow": MAX_ORD_FLOW, "cleaning_log": log,
               "rows": len(final), "columns": list(final.columns)}, f, indent=2, ensure_ascii=False)

size_mb = os.path.getsize(f"{OUT}/capybara_dataset.csv.gz") / 1e6
print(f"Ecrit : {OUT}/capybara_dataset.csv.gz ({size_mb:.1f} Mo)")

## Points à reporter dans le notebook principal

- Le DOI GBIF et les références de WorldClim et HydroRIVERS dans la description du projet.
- Le journal de nettoyage (section 4) et la justification des seuils.
- Les points sans valeur climatique et le choix fait pour eux.
- La carte : elle montre le biais d'échantillonnage vers les zones accessibles.
- `basis_of_record` et `coordinate_uncertainty_m` peuvent trahir la source plutôt que l'habitat. Comparez les modèles avec et sans.
- `spatial_cell` sert de groupe pour `StratifiedGroupKFold`, pas de variable d'entrée.